# TensorFlow & Keras: From Sequential to Custom Training

**Learning objectives:** Build small models with the Sequential and Functional APIs, run a minimal custom training loop with `tf.GradientTape`, inspect a frozen **MobileNetV2** base for transfer learning, and export a toy model with **SavedModel**.

## Introduction

**TensorFlow** is an end-to-end ML platform; **Keras** (now `tf.keras`) is its high-level API for defining layers, losses, and optimizers.

- **Sequential**: linear stack of layers—fast for simple MLPs and single-input nets.
- **Functional**: arbitrary DAGs—multiple inputs/outputs, shared weights, skip connections.
- **Subclassing** (not shown here): maximum flexibility at the cost of more boilerplate.

For full control over each batch, **`tf.GradientTape`** records forward ops so you can backprop manually. **Transfer learning** reuses weights trained on large data (e.g. ImageNet); **SavedModel** is TensorFlow’s portable bundle for serving and tooling.

In [ ]:
import numpy as np

try:
    import tensorflow as tf
except ImportError:
    print("Install: pip install tensorflow")
    raise

tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

### Sequential API: MLP on synthetic data

We regress a noisy linear target from random features—small, CPU-friendly, and enough to show `compile` / `fit`.

In [ ]:
rng = np.random.default_rng(42)
n_samples, n_features = 2_048, 16
X_syn = rng.normal(size=(n_samples, n_features)).astype("float32")
true_w = rng.normal(size=(n_features,)).astype("float32")
y_syn = (X_syn @ true_w + 0.1 * rng.normal(size=n_samples)).astype("float32")

seq_model = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(64, activation="relu", input_shape=(n_features,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1),
    ],
    name="sequential_mlp",
)
seq_model.compile(optimizer=tf.keras.optimizers.Adam(1e-2), loss="mse")
history = seq_model.fit(X_syn, y_syn, epochs=15, batch_size=64, verbose=0)
print("Final loss:", float(history.history["loss"][-1]))

### Functional API: multi-input model

Two inputs are concatenated before the head—this pattern maps to heterogeneous features.

In [ ]:
inp_a = tf.keras.Input(shape=(8,), name="branch_a")
inp_b = tf.keras.Input(shape=(8,), name="branch_b")
h1 = tf.keras.layers.Dense(16, activation="relu")(inp_a)
h2 = tf.keras.layers.Dense(16, activation="relu")(inp_b)
merged = tf.keras.layers.Concatenate()([h1, h2])
out = tf.keras.layers.Dense(1, activation="sigmoid")(merged)
multi_in = tf.keras.Model(inputs=[inp_a, inp_b], outputs=out, name="multi_input")
multi_in.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

Xa = rng.normal(size=(512, 8)).astype("float32")
Xb = rng.normal(size=(512, 8)).astype("float32")
yb = (Xa[:, 0] + Xb[:, 1] > 0).astype("float32")
h = multi_in.fit([Xa, Xb], yb, epochs=8, batch_size=32, verbose=0)
print("Final binary_crossentropy:", float(h.history["loss"][-1]))

### Custom training with `GradientTape`

The tape watches trainable variables; `gradient` yields grads for a manual `apply_gradients` step.

In [ ]:
dim = 12
w_star = rng.normal(size=(dim, 1)).astype("float32")
X_t = tf.constant(rng.normal(size=(256, dim)).astype("float32"))
y_t = tf.constant((X_t @ w_star + 0.05 * rng.normal(size=(256, 1))).astype("float32"))

simple = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(dim,))])
opt = tf.keras.optimizers.SGD(learning_rate=0.05)
loss_fn = tf.keras.losses.MeanSquaredError()

for step in range(200):
    with tf.GradientTape() as tape:
        pred = simple(X_t, training=True)
        loss = loss_fn(y_t, pred)
    grads = tape.gradient(loss, simple.trainable_variables)
    opt.apply_gradients(zip(grads, simple.trainable_variables))
    if step % 50 == 0:
        print(f"step {step:3d} mse={float(loss):.5f}")

### Transfer learning: MobileNetV2 (setup only)

We load **ImageNet** weights when available, freeze the base, and attach a small head—**no long fine-tune** here.

In [ ]:
try:
    base = tf.keras.applications.MobileNetV2(
        input_shape=(160, 160, 3),
        include_top=False,
        weights="imagenet",
    )
except Exception as exc:
    print("Imagenet weights unavailable (network/cache); using random init for structure demo.")
    print("Detail:", exc)
    base = tf.keras.applications.MobileNetV2(
        input_shape=(160, 160, 3), include_top=False, weights=None
    )

base.trainable = False
inputs = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
tl_model = tf.keras.Model(inputs, outputs, name="mobilenet_transfer_stub")
tl_model.summary()
print("Trainable variables:", sum(int(np.prod(v.shape)) for v in tl_model.trainable_variables))

### Model export: SavedModel

`tf.saved_model.save` writes a directory with signatures for TensorFlow Serving and other runtimes.

In [ ]:
import tempfile
from pathlib import Path

export_dir = Path(tempfile.mkdtemp(prefix="tf_savedmodel_"))
tf.saved_model.save(seq_model, str(export_dir))
print("SavedModel directory:", export_dir)
print("Sample entries:", [p.name for p in export_dir.iterdir()][:5])

## Conclusion

- Use **Sequential** for straightforward stacks; use **Functional** when you need multiple inputs/outputs or shared layers.
- **`GradientTape`** is the escape hatch when `fit` is not enough.
- **Transfer learning**: freeze a pretrained backbone first, add a small head, then optionally unfreeze top layers with a lower learning rate.
- **SavedModel** is the standard artifact for deployment—version export paths like any build output.